# 🌿 Project CAM — CRM Client Database Cleaning
## Technical Documentation — AgroTech Corp

---

| | |
|---|---|
| **Author** | S. Bougacha — M2 Data Science Internship |
| **Date** | May 2026 |
| **Language** | Python 3.11+ |
| **Libraries** | pandas · openpyxl · datetime · os |
| **Final output** | `client_analysis_final.xlsx` |

---

## 🎯 Objective

Clean a Salesforce CRM client database by cross-referencing **6 data sources** to automatically classify each account.

The pipeline processes **21,000+ accounts in seconds** and delivers a fully-colored, multi-sheet Excel report with detailed classification reasons for each account.

| Status | Meaning |
|---|---|
| 🟢 **Keep** | Confirmed active client |
| 🔴 **Deactivate** | No recent activity |
| 🟠 **Reintegrate** | Unconverted prospect to re-engage |

### Final Results
- 🟢 **Keep: 6,480** (30.5%)
- 🔴 **Deactivate: 14,633** (69.0%)
- 🟠 **Reintegrate: 97** (0.5%)
- **ERP Matching: 98.8%** (20,958 / 21,210 accounts matched)

---

## 🗂️ Data Sources

| File | Volume | Role |
|---|---|---|
| `accounts.csv` | 24,066 rows | Main source — status, dates, billing country |
| `opportunities.csv` | 10,472 rows | Sales opportunities per account |
| `invoice_history.csv` | 112,843 rows | Billing history |
| `active_invoices.csv` | 4,272 rows | Currently active invoices |
| `last_contact.xlsx` | 29,430 rows | Last CRM contact per account |
| `events.xlsx` | 42,659 rows | Last CRM event per account |
| `erp_clients.xlsx` | — | ERP client base for final matching |

---

## 🔄 Pipeline Overview — 5 Sequential Steps

```
Step 1: Initial Classification    (Accounts + Opportunities)
    ↓
Step 2: Invoice Enrichment        (History + Active invoices)
    ↓
Step 3: Contact & Event Check     (Last contact + Events)
    ↓
Step 4: Merge & Final Status      (Consolidation)
    ↓
Step 5: ERP Matching              (4-pass reliability-ordered matching)
```

---
## 📦 Imports & Configuration

In [ ]:
import pandas as pd
from datetime import date
import os
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

# ── Paths (relative — adapt to your environment) ──────────────────────────────
BASE_DIR   = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

today = date.today()

print(f'📅 Run date  : {today}')
print(f'📁 Data dir  : {DATA_DIR}')
print(f'📁 Output dir: {OUTPUT_DIR}')

---
## ⚙️ Shared Utility Functions

In [ ]:
# ── Robust CSV loader (auto-detects separator & encoding) ─────────────────────
def load_csv(path):
    """Load a CSV by auto-detecting separator and encoding."""
    for encoding in ['utf-8', 'latin-1']:
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(path, encoding=encoding, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass
    raise ValueError(f'Could not read: {path}')


# ── Date formatter for Excel export ──────────────────────────────────────────
def fmt_date(d):
    """Format a date as dd-MM-yyyy for Excel export."""
    if d is None or (isinstance(d, float) and pd.isna(d)):
        return ''
    try:
        return pd.Timestamp(d).strftime('%d-%m-%Y')
    except Exception:
        return str(d)


# ── Column reordering (ignores missing columns) ───────────────────────────────
def reorder_columns(df, priority_cols):
    """Place priority columns first; silently skip those absent from df."""
    existing = [c for c in priority_cols if c in df.columns]
    others   = [c for c in df.columns if c not in existing]
    return df[existing + others]


# ── Excel color palettes by classification status ─────────────────────────────
COLORS = {
    '🟢 Keep'         : {'fill': 'C6EFCE', 'font': '276221'},
    '🟡 New client'   : {'fill': 'FFEB9C', 'font': '9C6500'},
    '🟡 To monitor'   : {'fill': 'FFEB9C', 'font': '9C6500'},
    '🟠 Reintegrate'  : {'fill': 'FCE4D6', 'font': '974706'},
    '🔴 To review'    : {'fill': 'FFC7CE', 'font': '9C0006'},
    '🔴 Deactivate'   : {'fill': 'FFC7CE', 'font': '9C0006'},
}

COLORS_MATCH = {
    'Yes' : {'fill': 'C6EFCE', 'font': '276221'},
    'No'  : {'fill': 'FFC7CE', 'font': '9C0006'},
}

COLORS_VIA = {
    'ERP ID'           : {'fill': '1F4E79', 'font': 'FFFFFF'},
    'Fusion Code 1 & 2': {'fill': '375623', 'font': 'FFFFFF'},
    'Fusion Code 1'    : {'fill': '70AD47', 'font': 'FFFFFF'},
    'Fusion Code 2'    : {'fill': 'A9D18E', 'font': '375623'},
    'Client name'      : {'fill': 'FFD966', 'font': '7F6000'},
    'No match found'   : {'fill': 'FFC7CE', 'font': '9C0006'},
}


# ── Apply color formatting to an Excel sheet ──────────────────────────────────
def color_sheet(ws):
    """Apply conditional coloring on Status, ERP_Match, and Match_via columns."""
    header     = [cell.value for cell in ws[1]]
    col_status = header.index('Final_Status') + 1 if 'Final_Status' in header else None
    col_match  = header.index('ERP_Match')    + 1 if 'ERP_Match'    in header else None
    col_via    = header.index('Match_via')    + 1 if 'Match_via'    in header else None

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        if col_status:
            cell = row[col_status - 1]
            val  = str(cell.value) if cell.value else ''
            for key, c in COLORS.items():
                if key in val or val in key:
                    cell.fill = PatternFill(start_color=c['fill'], end_color=c['fill'], fill_type='solid')
                    cell.font = Font(color=c['font'], bold=True)
                    break
        if col_match:
            cell = row[col_match - 1]
            val  = str(cell.value) if cell.value else ''
            if val in COLORS_MATCH:
                c = COLORS_MATCH[val]
                cell.fill = PatternFill(start_color=c['fill'], end_color=c['fill'], fill_type='solid')
                cell.font = Font(color=c['font'], bold=True)
        if col_via:
            cell = row[col_via - 1]
            val  = str(cell.value) if cell.value else ''
            if val in COLORS_VIA:
                c = COLORS_VIA[val]
                cell.fill = PatternFill(start_color=c['fill'], end_color=c['fill'], fill_type='solid')
                cell.font = Font(color=c['font'], bold=True)

print('✅ Utility functions loaded')

---
## 📊 Step 1 — Initial Classification (Accounts + Opportunities)

### Logic
```
Accounts  ──LEFT JOIN──▶  Opportunities
   Id                      AccountId
```

7 priority rules applied in order — first match wins.

| Priority | Condition | Status |
|---|---|---|
| P1 | Prospect + Target Country + Created ≤ 2023 | 🟠 Reintegrate |
| P2 | Status = Customer | 🟢 Keep |
| P3 | Status = Old Customer + no opp OR inactive 5+ yrs | 🔴 Review |
| P4 | Status = Not Applicable or Prospect (other) | 🔴 Review |
| P5 | No opportunity + account < 1 yr old | 🟡 New client |
| P6 | Last activity > 5 years ago | 🔴 Deactivate |
| P7 | All other cases (recent confirmed activity) | 🟢 Keep |

### Step 1 Results
| Status | Count |
|---|---|
| 🔴 To review | 19,750 |
| 🟢 Keep | 1,363 |
| 🟠 Reintegrate | 97 |

In [ ]:
# ─── 1. LOAD ─────────────────────────────────────────────────────────────────
accounts      = pd.read_csv(os.path.join(DATA_DIR, 'accounts.csv'), encoding='utf-8', low_memory=False)
opportunities = pd.read_csv(os.path.join(DATA_DIR, 'opportunities.csv'), encoding='utf-8', low_memory=False)

accounts      = accounts.loc[:, ~accounts.columns.str.startswith('Unnamed')]
opportunities = opportunities.loc[:, ~opportunities.columns.str.startswith('Unnamed')]

print(f'✅ Accounts loaded      : {len(accounts):,} rows')
print(f'✅ Opportunities loaded : {len(opportunities):,} rows')

In [ ]:
# ─── 2. FILTER closed accounts ───────────────────────────────────────────────
# Accounts flagged AccountClosed__c = True are excluded from scope
before   = len(accounts)
accounts = accounts[accounts['AccountClosed__c'].astype(str).str.lower() != 'true']
print(f'✅ Closed accounts excluded: {before - len(accounts):,} | Remaining: {len(accounts):,}')

In [ ]:
# ─── 3. DATE PARSING ─────────────────────────────────────────────────────────
for col in ['CreatedDate', 'LastModifiedDate']:
    if col in accounts.columns:
        accounts[col] = pd.to_datetime(accounts[col], errors='coerce').dt.date
    if col in opportunities.columns:
        opportunities[col] = pd.to_datetime(opportunities[col], errors='coerce').dt.date
print('✅ Dates parsed')

In [ ]:
# ─── 4. OPPORTUNITY AGGREGATION per account ───────────────────────────────────
opp_agg = opportunities.groupby('AccountId').agg(
    nb_opportunities  = ('Id', 'count'),
    last_opp_stage    = ('StageName', 'last'),
    last_opp_modified = ('LastModifiedDate', 'max'),
    last_opp_created  = ('CreatedDate', 'max'),
).reset_index()

# ─── 5. LEFT JOIN Accounts × Opportunities ───────────────────────────────────
df = accounts.merge(opp_agg, left_on='Id', right_on='AccountId', how='left')
df.drop(columns=['AccountId'], errors='ignore', inplace=True)
print(f'✅ Join complete: {len(df):,} rows')

In [ ]:
# ─── 6. LAST ACTIVITY DATE ───────────────────────────────────────────────────
# last_activity_date = max(account LastModifiedDate, last opportunity modified date)
def max_date(row):
    dates = [row.get('LastModifiedDate'), row.get('last_opp_modified')]
    dates = [d for d in dates if d is not None and pd.notna(d)]
    return max(dates) if dates else None

df['last_activity_date'] = df.apply(max_date, axis=1)
print('✅ last_activity_date computed')

In [ ]:
# ─── 7. CLASSIFICATION — 7 priority rules ────────────────────────────────────
# NOTE: 'target_country' below represents a specific country flag used as a
# business re-engagement criterion. Adapt to your own use case.
TARGET_COUNTRY_TERMS = ['brazil', 'brasil', 'br']  # customize as needed

def classify(row):
    has_opp  = pd.notna(row.get('nb_opportunities')) and row['nb_opportunities'] > 0
    created  = row.get('CreatedDate')
    last_act = row.get('last_activity_date')
    status   = str(row.get('Status__c', '')).strip().lower()
    country  = str(row.get('BillingCountry', '')).strip().lower()

    age_years = (today - created).days  / 365 if pd.notna(created)  else None
    act_years = (today - last_act).days / 365 if pd.notna(last_act) else None

    # P1 — Target-country prospect created before threshold year
    if (status == 'prospect'
            and pd.notna(created) and created.year <= 2023
            and any(x in country for x in TARGET_COUNTRY_TERMS)):
        return '🟠 Reintegrate', 'Unconverted target-country prospect, created 2023 or earlier'

    # P2 — Active customer → always keep
    if status == 'customer':
        return '🟢 Keep', 'Confirmed active client (Customer status)'

    # P3 — Former customer
    if status == 'old customer':
        if not has_opp or (act_years is not None and act_years > 5):
            return '🔴 To review', 'Former client with no recent opportunity (inactive 5+ yrs)'
        return '🟢 Keep', f'Former client with recent activity (last: {last_act})'

    # P4 — Not Applicable or non-target Prospect
    if status in ('not applicable', 'prospect'):
        return '🔴 To review', f'Unqualified status ({row.get("Status__c", "")})'

    # P5 — No opportunity
    if not has_opp:
        if age_years is not None and age_years < 1:
            return '🟡 New client', 'Recent account (< 1 yr) — no opportunity yet'
        return '🔴 To review', 'No opportunity on record — manual review required'

    # P6 — Inactive for 5+ years
    if act_years is not None and act_years > 5:
        return '🔴 Deactivate', f'Inactive for 5+ years (last activity: {last_act})'

    # P7 — Active by default
    return '🟢 Keep', f'Recent activity detected (last: {last_act})'


df[['Final_Status', 'Reason']] = df.apply(lambda row: pd.Series(classify(row)), axis=1)
print('✅ Classification complete')
print('\n📊 Distribution:')
print(df['Final_Status'].value_counts().to_string())

In [ ]:
# ─── 8. EXPORT ───────────────────────────────────────────────────────────────
for col in ['CreatedDate','LastModifiedDate','last_opp_modified','last_opp_created','last_activity_date']:
    if col in df.columns:
        df[col] = df[col].apply(fmt_date)

PRIORITY_COLS_1 = [
    'Id','Name','Local_Name__c','Status__c','BillingCountry',
    'CreatedDate','LastModifiedDate',
    'nb_opportunities','last_opp_stage','last_opp_modified','last_opp_created','last_activity_date',
    'Final_Status','Reason',
    'Owner.Name','Sage_ID__c','Code_Fusion__c','Company__r.Sage_ID__c','AccountClosed__c'
]
df = reorder_columns(df, PRIORITY_COLS_1)

output_path = os.path.join(OUTPUT_DIR, 'step1_initial_classification.xlsx')
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Full analysis', index=False)
    for status in df['Final_Status'].unique():
        df[df['Final_Status']==status].to_excel(writer, sheet_name=status[:28].replace('/','−'), index=False)
    (df['Final_Status'].value_counts()
     .reset_index().rename(columns={'Final_Status':'Status','count':'Count'})
     .to_excel(writer, sheet_name='Summary', index=False))

wb = load_workbook(output_path)
for sn in wb.sheetnames:
    color_sheet(wb[sn])
wb.save(output_path)
print(f'✅ Step 1 exported: {output_path}')

---
## 💳 Step 2 — Invoice Enrichment

### Objective
Reclassify the **19,750 accounts** initially flagged as 🔴 To review using billing history.

### Join logic
```
Analysis V1  ──LEFT JOIN──▶  invoice_history.csv
FusionCode                    BUSINESS PARTNER N°

             ──LOOKUP──▶     active_invoices.csv
FusionCode                    BUSINESS PARTNER N° (presence = True)
```

### Reclassification rules
| Condition | New status |
|---|---|
| Active invoice present | 🟢 Keep |
| Last invoice < 3 years | 🟢 Keep |
| Last invoice 3–5 years | 🟡 To monitor |
| Last invoice > 5 years or absent | 🔴 Deactivate |

### Step 2 Results
| Status | Count |
|---|---|
| 🔴 Deactivate | 19,372 |
| 🟢 Keep | 1,505 |
| 🟡 To monitor | 236 |
| 🟠 Reintegrate | 97 |

In [ ]:
today_ts = pd.Timestamp(today)

# ─── 1. LOAD ─────────────────────────────────────────────────────────────────
df = pd.read_excel(os.path.join(OUTPUT_DIR, 'step1_initial_classification.xlsx'), sheet_name='Full analysis')
print(f'✅ Step 1 analysis loaded: {len(df):,} rows')
print(f'   Of which To review: {(df["Final_Status"]=="🔴 To review").sum():,}')

invoice_history = load_csv(os.path.join(DATA_DIR, 'invoice_history.csv'))
active_invoices = load_csv(os.path.join(DATA_DIR, 'active_invoices.csv'))
invoice_history.columns = invoice_history.columns.str.strip()
active_invoices.columns = active_invoices.columns.str.strip()
print(f'✅ Invoice history: {len(invoice_history):,} rows | Active invoices: {len(active_invoices):,} rows')

In [ ]:
# ─── 2. INVOICE AGGREGATION ──────────────────────────────────────────────────
invoice_history['INVOICE DATE'] = pd.to_datetime(
    invoice_history['INVOICE DATE'], errors='coerce', dayfirst=True).dt.date
active_invoices['INVOICE DATE'] = pd.to_datetime(
    active_invoices['INVOICE DATE'], errors='coerce', dayfirst=True).dt.date

hist_agg = (
    invoice_history.groupby('BUSINESS PARTNER N°')['INVOICE DATE']
    .max().reset_index()
    .rename(columns={'INVOICE DATE': 'last_invoice_date', 'BUSINESS PARTNER N°': 'bp_num'})
)
active_set = set(active_invoices['BUSINESS PARTNER N°'].dropna().astype(str).str.strip().unique())

# ─── 3. JOIN ─────────────────────────────────────────────────────────────────
df['_key']       = df['Code_Fusion__c'].astype(str).str.strip()
hist_agg['_key'] = hist_agg['bp_num'].astype(str).str.strip()
df = df.merge(hist_agg[['_key', 'last_invoice_date']], on='_key', how='left')
df.drop(columns=['_key'], errors='ignore', inplace=True)
df['has_active_invoice'] = df['Code_Fusion__c'].astype(str).str.strip().isin(active_set)

print(f'✅ Matched to history: {df["last_invoice_date"].notna().sum():,} | Active invoice: {df["has_active_invoice"].sum():,}')

In [ ]:
# ─── 4. RECLASSIFICATION ─────────────────────────────────────────────────────
def reclassify_invoices(row):
    if row['Final_Status'] != '🔴 To review':
        return row['Final_Status'], row['Reason']

    last_inv = row.get('last_invoice_date')

    if row.get('has_active_invoice'):
        return '🟢 Keep', 'Active invoice currently being paid'

    if pd.notna(last_inv) and last_inv is not None:
        inv_years = (today - last_inv).days / 365
        if inv_years < 3:
            return '🟢 Keep', f'Recent payment (last invoice: {last_inv.strftime("%d-%m-%Y")})'
        if inv_years <= 5:
            return '🟡 To monitor', f'Financial activity between 3–5 years (last invoice: {last_inv.strftime("%d-%m-%Y")})'
        return '🔴 Deactivate', f'No financial activity for 5+ years (last invoice: {last_inv.strftime("%d-%m-%Y")})'

    return '🔴 Deactivate', 'Absent from invoice history and active invoices'


df[['Final_Status', 'Reason']] = df.apply(lambda row: pd.Series(reclassify_invoices(row)), axis=1)
df['last_invoice_date'] = df['last_invoice_date'].apply(fmt_date)

print('✅ Reclassification complete')
print('\n📊 Distribution:')
print(df['Final_Status'].value_counts().to_string())

---
## 📞 Step 3 — Contact & Event Verification

### Objective
Among the **19,372 accounts** flagged 🔴 Deactivate, rescue any with recent CRM activity (< 3 years).

### Join logic
```
To deactivate  ──LEFT JOIN──▶  last_contact.xlsx   →  last_contact_date
Id                              AccountId

               ──LEFT JOIN──▶  events.xlsx          →  last_event_date
Id                              AccountId
```

### Rule: if ANY criterion detects activity < 3 years → 🟢 Keep

| Criterion | Threshold | Action |
|---|---|---|
| Last contact | < 3 years | 🟢 Keep |
| Last event | < 3 years | 🟢 Keep |
| Account age | < 3 years | 🟢 Keep |
| No criterion met | — | 🔴 Confirmed Deactivate |

### Step 3 Results
| Status | Count |
|---|---|
| 🔴 Confirmed Deactivate | 14,633 |
| 🟢 Reclassified Keep | 4,739 |

In [ ]:
# ─── 1. LOAD ─────────────────────────────────────────────────────────────────
df_full  = pd.read_excel(os.path.join(OUTPUT_DIR, 'step2_invoice_enrichment.xlsx'), sheet_name='Full analysis')
contacts = pd.read_excel(os.path.join(DATA_DIR, 'last_contact.xlsx'))
events   = pd.read_excel(os.path.join(DATA_DIR, 'events.xlsx'))

for frame in [contacts, events]:
    frame.drop(columns=[c for c in frame.columns if c.startswith('Unnamed')], inplace=True, errors='ignore')
    frame.columns = frame.columns.str.strip()

# Keep as datetime64 for groupby max() compatibility
contacts['CreatedDate'] = pd.to_datetime(contacts['CreatedDate'], errors='coerce', utc=True).dt.tz_localize(None)
events['CreatedDate']   = pd.to_datetime(events['CreatedDate'],   errors='coerce', utc=True).dt.tz_localize(None)

print(f'✅ Contacts: {len(contacts):,} rows | Events: {len(events):,} rows')

In [ ]:
# ─── 2. FOCUS + JOINS ────────────────────────────────────────────────────────
contact_agg = (contacts.groupby('AccountId')['CreatedDate'].max()
               .reset_index().rename(columns={'CreatedDate': 'last_contact_date'}))
event_agg   = (events.groupby('AccountId')['CreatedDate'].max()
               .reset_index().rename(columns={'CreatedDate': 'last_event_date'}))

to_deactivate = df_full[df_full['Final_Status'] == '🔴 Deactivate'].copy()
to_deactivate = (to_deactivate
    .merge(contact_agg, left_on='Id', right_on='AccountId', how='left').drop(columns=['AccountId'], errors='ignore')
    .merge(event_agg,   left_on='Id', right_on='AccountId', how='left').drop(columns=['AccountId'], errors='ignore'))

print(f'✅ Accounts to deactivate: {len(to_deactivate):,}')
print(f'   With contact: {to_deactivate["last_contact_date"].notna().sum():,} | With event: {to_deactivate["last_event_date"].notna().sum():,}')

In [ ]:
# ─── 3. ACCOUNT AGE ──────────────────────────────────────────────────────────
to_deactivate['_created_parsed'] = pd.to_datetime(to_deactivate['CreatedDate'], dayfirst=True, errors='coerce')
to_deactivate['account_age_years'] = to_deactivate['_created_parsed'].apply(
    lambda d: round((today_ts - d).days / 365, 1) if pd.notna(d) else None
)

# ─── 4. RECLASSIFICATION ─────────────────────────────────────────────────────
def reclassify_deactivate(row):
    age     = row.get('account_age_years')
    lc      = row.get('last_contact_date')
    le      = row.get('last_event_date')
    c_years = (today_ts - lc).days / 365 if pd.notna(lc) else None
    e_years = (today_ts - le).days / 365 if pd.notna(le) else None

    if c_years is not None and c_years < 3:
        return '🟢 Keep', f'Recent contact detected (last: {lc.strftime("%d-%m-%Y")})'
    if e_years is not None and e_years < 3:
        return '🟢 Keep', f'Recent event detected (last: {le.strftime("%d-%m-%Y")})'
    if age is not None and age < 3:
        return '🟢 Keep', f'Recent account ({age} yrs) — monitor before deactivation'

    parts = []
    parts.append(f'account created {age} yrs ago' if age else 'unknown age')
    parts.append(f'last contact {round(c_years,1)} yrs ago' if c_years else 'no contact')
    parts.append(f'last event {round(e_years,1)} yrs ago' if e_years else 'no event')
    return '🔴 Deactivate', 'Confirmed — ' + ' | '.join(parts)


to_deactivate[['Final_Status', 'Reason']] = to_deactivate.apply(
    lambda row: pd.Series(reclassify_deactivate(row)), axis=1)
to_deactivate['last_contact_date'] = to_deactivate['last_contact_date'].apply(fmt_date)
to_deactivate['last_event_date']   = to_deactivate['last_event_date'].apply(fmt_date)
to_deactivate.drop(columns=['_created_parsed'], errors='ignore', inplace=True)

print('✅ Reclassification complete')
print('\n📊 Results:')
print(to_deactivate['Final_Status'].value_counts().to_string())

---
## 🔀 Step 4 — Merge & Final ADEL Results

### Logic
```
Step 2 analysis                 Step 3 focus
(without 🔴 Deactivate)   +   (14,633 confirmed + 4,739 reclassified)
         └─────────── pd.concat ────────────┘
                           ↓
         Dedup on Id (Step 3 takes priority)
                           ↓
    🟡 New client  →  🟢 Keep  (reason preserved)
    🟡 To monitor  →  🟢 Keep
```

### Final ADEL Results
| Status | Count | % |
|---|---|---|
| 🟢 Keep | 6,480 | 30.5% |
| 🔴 Deactivate | 14,633 | 69.0% |
| 🟠 Reintegrate | 97 | 0.5% |

In [ ]:
# ─── 1. MERGE ────────────────────────────────────────────────────────────────
analysis_main  = pd.read_excel(os.path.join(OUTPUT_DIR, 'step2_invoice_enrichment.xlsx'), sheet_name='Full analysis')
analysis_focus = to_deactivate  # already in memory from step 3

df_final = pd.concat([
    analysis_main[analysis_main['Final_Status'] != '🔴 Deactivate'],
    analysis_focus
], ignore_index=True)

dupes = df_final['Id'].duplicated().sum()
if dupes > 0:
    df_final = df_final.drop_duplicates(subset='Id', keep='last')
    print(f'⚠️  {dupes} duplicate(s) removed')

# ─── 2. COLLAPSE INTERMEDIATE STATUSES ───────────────────────────────────────
# 🟡 statuses (New client, To monitor) → 🟢 Keep (reason is preserved)
df_final['Final_Status'] = df_final['Final_Status'].apply(
    lambda s: '🟢 Keep' if '🟡' in str(s) else s
)

print(f'✅ Merge complete: {len(df_final):,} rows')
print('\n📊 Final distribution:')
print(df_final['Final_Status'].value_counts().to_string())

---
## 🔗 Step 5 — ERP Matching

### Objective
Match all 21,210 CRM accounts against the ERP client base via **4 reliability-ordered passes**.

### Key innovation — Bidirectional Fusion Code matching
The fusion code (`Code_Fusion__c`) is tested simultaneously against both `FUSION_CODE_1` **and** `FUSION_CODE_2` in the ERP. Three possible outcomes:
- **Both codes**: present in both ERP columns → detail stored in `ERP_Fusion_Detail`
- **Code 1 only**: matched via `FUSION_CODE_1`
- **Code 2 only**: matched via `FUSION_CODE_2`

This innovation drove coverage from **8.3% → 98.8%**.

### Matching passes (in reliability order)
| Pass | CRM key | ERP key | Matches | Coverage |
|---|---|---|---|---|
| T1 | Account Id | ERP_ID | 4,175 | 20% |
| T2 | Fusion Code | FUSION_CODE_1 & 2 | 16,675 | 79.6% |
| T3 | Fusion Code | FUSION_CODE_2 only | 70 | 0.3% |
| T4 | Account Name | CLIENT_NAME | 38 | 0.2% |

### Final results by status
| Status | Matched | Unmatched | % Matched |
|---|---|---|---|
| 🟢 Keep | 6,390 | 90 | 98.6% |
| 🔴 Deactivate | 14,483 | 150 | 99.0% |
| 🟠 Reintegrate | 85 | 12 | 87.6% |
| **TOTAL** | **20,958** | **252** | **98.8%** |

In [ ]:
# ─── 1. LOAD ─────────────────────────────────────────────────────────────────
df   = pd.read_excel(os.path.join(OUTPUT_DIR, 'step4_final_crm.xlsx'), sheet_name='Full analysis')
erp  = pd.read_excel(os.path.join(DATA_DIR, 'erp_clients.xlsx'))
erp  = erp.loc[:, ~erp.columns.str.startswith('Unnamed')]
erp.columns = erp.columns.str.strip()

print(f'✅ CRM analysis: {len(df):,} rows | Keep: {(df["Final_Status"]=="🟢 Keep").sum():,}')
print(f'✅ ERP: {len(erp):,} rows | Columns: {list(erp.columns)}')

In [ ]:
# ─── 2. NORMALIZATION ────────────────────────────────────────────────────────
EMPTY_VALUES = {'NAN', 'NONE', '', ' '}

df['_id']          = df['Id'].astype(str).str.strip().str.upper()
df['_fusion_code'] = df['Code_Fusion__c'].astype(str).str.strip().str.upper()
df['_name']        = df['Name'].astype(str).str.strip().str.upper()

erp['_erp_id']    = erp['ERP_ID'].astype(str).str.strip().str.upper()
erp['_fusion1']   = erp['FUSION_CODE_1'].astype(str).str.strip().str.upper()
erp['_fusion2']   = erp['FUSION_CODE_2'].astype(str).str.strip().str.upper()
erp['_name']      = erp['CLIENT_NAME'].astype(str).str.strip().str.upper()

print('✅ Keys normalized')

In [ ]:
# ─── 3. LOOKUP INDEXES ───────────────────────────────────────────────────────
lookup_erp_id = erp[~erp['_erp_id'].isin(EMPTY_VALUES)].drop_duplicates('_erp_id').set_index('_erp_id')
lookup_f1     = erp[~erp['_fusion1'].isin(EMPTY_VALUES)].drop_duplicates('_fusion1').set_index('_fusion1')
lookup_f2     = erp[~erp['_fusion2'].isin(EMPTY_VALUES)].drop_duplicates('_fusion2').set_index('_fusion2')
lookup_name   = erp[~erp['_name'].isin(EMPTY_VALUES)].drop_duplicates('_name').set_index('_name')

set_f1 = set(lookup_f1.index)
set_f2 = set(lookup_f2.index)

print(f'✅ Lookup ERP_ID: {len(lookup_erp_id):,} | Fusion1: {len(lookup_f1):,} | Fusion2: {len(lookup_f2):,} | Name: {len(lookup_name):,}')

In [ ]:
# ─── 4. ERP MATCHING — 4 reliability-ordered passes ──────────────────────────
def match_erp(row):
    aid = row['_id']
    cf  = row['_fusion_code']
    nm  = row['_name']

    # T1 — Direct ERP ID (highest reliability)
    if aid not in EMPTY_VALUES and aid in lookup_erp_id.index:
        r = lookup_erp_id.loc[aid]
        return 'Yes', 'ERP ID', str(r.get('ERP_CLIENT_CODE', '')), str(r.get('CLIENT_LEGAL_NAME', '')), ''

    # T2 — Fusion code (bidirectional: FUSION_CODE_1 AND/OR FUSION_CODE_2)
    if cf not in EMPTY_VALUES:
        in_f1, in_f2 = cf in set_f1, cf in set_f2

        if in_f1 and in_f2:  # Double match — both columns
            r1, r2 = lookup_f1.loc[cf], lookup_f2.loc[cf]
            c1 = str(r1.get('ERP_CLIENT_CODE', ''))
            c2 = str(r2.get('ERP_CLIENT_CODE', ''))
            return 'Yes', 'Fusion Code 1 & 2', c1, str(r1.get('CLIENT_LEGAL_NAME', '')), f'F1:{c1} | F2:{c2}'
        elif in_f1:
            r = lookup_f1.loc[cf]
            return 'Yes', 'Fusion Code 1', str(r.get('ERP_CLIENT_CODE', '')), str(r.get('CLIENT_LEGAL_NAME', '')), ''
        elif in_f2:
            r = lookup_f2.loc[cf]
            return 'Yes', 'Fusion Code 2', str(r.get('ERP_CLIENT_CODE', '')), str(r.get('CLIENT_LEGAL_NAME', '')), ''

    # T3 — Client name (least reliable)
    if nm not in EMPTY_VALUES and nm in lookup_name.index:
        r = lookup_name.loc[nm]
        return 'Yes', 'Client name', str(r.get('ERP_CLIENT_CODE', '')), str(r.get('CLIENT_LEGAL_NAME', '')), ''

    return 'No', 'No match found', '', '', ''


df[['ERP_Match', 'Match_via', 'ERP_Code', 'ERP_Legal_Name', 'ERP_Fusion_Detail']] = df.apply(
    lambda row: pd.Series(match_erp(row)), axis=1
)
df.drop(columns=['_id', '_fusion_code', '_name'], inplace=True)

print('✅ ERP matching complete')
print(f'\n📊 Overall results:')
print(df['ERP_Match'].value_counts().to_string())
print(f'\n📊 Detail by pass:')
print(df[df['ERP_Match']=='Yes']['Match_via'].value_counts().to_string())
print(f'\n📊 Keep accounts match rate:')
print(df[df['Final_Status']=='🟢 Keep']['ERP_Match'].value_counts().to_string())

In [ ]:
# ─── 5. SUMMARY TABLES ───────────────────────────────────────────────────────
summary_crosstab = pd.crosstab(
    df['Final_Status'], df['ERP_Match'], margins=True, margins_name='Total'
).reset_index()

summary_passes = df[df['ERP_Match']=='Yes']['Match_via'].value_counts().reset_index()
summary_passes.columns = ['Match pass', 'Account count']

summary_rate = df.groupby('Final_Status')['ERP_Match'].value_counts().unstack(fill_value=0).reset_index()
summary_rate['Total']      = summary_rate.get('Yes', 0) + summary_rate.get('No', 0)
summary_rate['% Matched']  = (summary_rate.get('Yes', 0) / summary_rate['Total'] * 100).round(1).astype(str) + '%'

print('✅ Summaries computed')
print('\n📊 Match rate by status:')
print(summary_rate[['Final_Status', 'Yes', 'No', 'Total', '% Matched']].to_string(index=False))

In [ ]:
# ─── 6. FINAL EXPORT ─────────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, 'client_analysis_final.xlsx')

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Full analysis', index=False)

    for status in ['🟢 Keep', '🔴 Deactivate', '🟠 Reintegrate']:
        subset = df[df['Final_Status'] == status]
        if len(subset) > 0:
            subset.to_excel(writer, sheet_name=status[:28].replace('/', '−'), index=False)

    df[df['ERP_Match'] == 'No'].to_excel(writer, sheet_name='Unmatched ERP', index=False)
    df[(df['ERP_Match'] == 'No') & (df['Final_Status'] == '🟢 Keep')].to_excel(
        writer, sheet_name='Keep unmatched ERP', index=False)

    (df['Final_Status'].value_counts()
     .reset_index().rename(columns={'Final_Status': 'Status', 'count': 'Count'})
     .to_excel(writer, sheet_name='Summary', index=False))

    summary_crosstab.to_excel(writer, sheet_name='ERP Matching Summary', index=False, startrow=1)
    ws = writer.sheets['ERP Matching Summary']
    ws.cell(row=1, column=1, value='Status × ERP Match breakdown')
    off1 = len(summary_crosstab) + 4
    ws.cell(row=off1, column=1, value='Detail by matching pass')
    summary_passes.to_excel(writer, sheet_name='ERP Matching Summary', index=False, startrow=off1)
    off2 = off1 + len(summary_passes) + 4
    ws.cell(row=off2, column=1, value='Match rate by status')
    summary_rate.to_excel(writer, sheet_name='ERP Matching Summary', index=False, startrow=off2)

# Apply color formatting
TITLE_FILL = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
TITLE_FONT = Font(bold=True, color='1F3864')
wb = load_workbook(output_path)
for sn in wb.sheetnames:
    if sn not in ['Summary', 'ERP Matching Summary']:
        color_sheet(wb[sn])
for row in wb['ERP Matching Summary'].iter_rows():
    for cell in row:
        if isinstance(cell.value, str) and any(x in cell.value for x in ['breakdown', 'Detail', 'rate']):
            cell.fill = TITLE_FILL
            cell.font = TITLE_FONT
wb.save(output_path)

print(f'\n✅ FINAL FILE EXPORTED: {output_path}')
print(f'\n📊 Final distribution:')
print(df['Final_Status'].value_counts().to_string())
total = len(df)
matched = df['ERP_Match'].value_counts()['Yes']
print(f'\n📊 ERP match rate: {matched:,} / {total:,} = {matched/total*100:.1f}%')

---
## 📋 Final Summary

| Step | Action | Input | Output |
|---|---|---|---|
| **1** | Initial classification | Accounts + Opportunities | 19,750 flagged for review |
| **2** | Invoice enrichment | History + Active invoices | 19,372 to deactivate |
| **3** | Contact & Event check | 19,372 to deactivate | 14,633 confirmed |
| **4** | Merge & consolidation | Steps 2 + 3 | 6,480 / 14,633 / 97 |
| **5** | ERP matching | ERP client base | **98.8% matched** |

### 🟢 Final output: `client_analysis_final.xlsx`
9 sheets · Conditional color formatting · Detailed reasons · Full traceability

---
*AgroTech Corp — M2 Data Science Internship — May 2026*